# HSLS:09 Dropout Prediction — Data Cleaning

## Purpose and Scope

This notebook documents all data cleaning decisions applied to the HSLS:09 public-use 
file before any modelling takes place. The dataset has been explored in a previous 
notebook and the cleaning decisions are made based on the findings during that 
exploration. Each decision is stated explicitly and verified through code output. The 
resulting clean dataset is saved as a single CSV file serving as input for all 
subsequent modelling notebooks.

Three sample definitions are used consistently throughout this notebook:

- **Full sample** — all 23,503 students sampled by NCES in the base year, regardless 
  of response status.
- **Base year respondents** — the 21,444 students who completed the base year 
  questionnaire. All wave-level counts and dropout mappings are drawn from this group.
- **Analysis sample** — the final set of students satisfying all inclusion criteria: 
  base year response and valid X4EVERDROP. This is the dataset that enters the model.

---

## Research Goal

The objective is to predict which students are at risk of dropping out of high school 
using only information available at the start of 9th grade, enabling institutions to 
direct early intervention resources toward at-risk students before a dropout episode 
occurs.

This imposes a strict constraint: **all features must come from the base year (fall 
2009)**. Any variable measured after the base year would constitute look-ahead bias — 
the model would appear to perform well in training but fail in real deployment where 
future data does not yet exist.

---

## Target Variable

The HSLS:09 dataset tracks students across four survey waves spanning from 9th grade 
in 2009 through young adulthood in 2016. The target variable is `X4EVERDROP` — a 
binary indicator of whether a student ever dropped out of high school, measured at 
the Second Follow-Up (F2) in 2016, approximately three years after the students' expected 
graduation date. This wave is chosen because it provides the most 
complete picture of dropout across the entire high school period. A student with a confirmed dropout episode at any point between 2009 and 2016 is captured as X4EVERDROP = 1, regardless of whether they later returned to school or obtained a GED.

# Data Cleaning

## Why Intermediate Wave Data Is Excluded as Features

The intermediate waves F1 and U13 captured student circumstances two to four years 
into high school. The relevant variables carry either an X2 or X3 prefix and are 
excluded as features for the following reason: they partially describe the outcome 
we are trying to predict — a student's enrollment status or engagement in 2012 is 
not independent of whether they eventually drop out, it is part of the dropout 
process itself.

Importantly, the intermediate wave data in the HSLS:09 dataset is used to construct 
and verify X4EVERDROP. We therefore examine the intermediate variables to understand 
how X4EVERDROP is built. The variables from the intermediate waves are dropped 
entirely from the clean dataset.

---

## Sample Construction

The analysis sample is built by applying two sequential conditions to the full sample.

**Condition 1 — Base year response:** The student completed the base year 
questionnaire (X1SQSTAT in 1, 2, 3). Students coded 8 are non-respondents with no 
feature data.

| Code | Description | Count |
|------|-------------|-------|
| 1 | In-school self-administered form | 21,001 |
| 2 | Out-of-school self-administered form | 29 |
| 3 | CATI — telephone interview | 414 |
| 8 | Non-respondent | 2,059 |
| **Total** | | **23,503** |

Codes 1, 2, and 3 are treated equivalently — all represent a completed questionnaire 
regardless of administration mode. Applying this condition reduces the full sample 
from 23,503 to **21,444 base year respondents**.

**Condition 2 — Valid outcome at F2:** The student has X4EVERDROP = 0 or 1. Two 
exclusions apply here. First, 6,168 students coded X4SQSTAT = 8 did not participate 
in F2 at all. Second, three students completed F2 (X4SQSTAT in 1, 2, 3, 4) but 
still have X4EVERDROP = -9 — their responses to both the dropout question and the 
prior wave indicator were inconclusive, meaning the construction logic could not 
resolve their status. Both groups are excluded.

| Code | Description | Count |
|------|-------------|-------|
| 1 | Full-length web instrument | 12,940 |
| 2 | Full-length CATI | 2,925 |
| 3 | Abbreviated web instrument | 955 |
| 4 | Abbreviated CATI | 515 |
| 8 | Non-respondent | 6,168 |
| **Total** | | **23,503** |

Applying conditions 1 and 2 together defines four valid response patterns across the 
four survey waves (Base Year — F1 — U13 — F2), where 1 = responded, 0 = did not 
respond. This reduces the base year respondents from 21,444 to **15,906 students**:

| Pattern | Description | Count |
|---------|-------------|-------|
| 1-1-1-1 | Responded at all waves | 13,280 |
| 1-1-0-1 | Responded BY, F1, F2 — skipped U13 | 1,430 |
| 1-0-1-1 | Responded BY, U13, F2 — skipped F1 | 797 |
| 1-0-0-1 | Responded BY and F2 only | 399 |
| **Total** | | **15,906** |

---

Note on potential bias: the 5,535 base year respondents lost to F2 non-response are 
not randomly distributed. Students facing greater socioeconomic instability and 
disengagement are more likely to both drop out of high school and drop out of 
longitudinal surveys. The analysis sample therefore likely underrepresents the most 
at-risk students, and the observed dropout rate likely underestimates the true rate 
in the full sample. This limitation must be acknowledged when generalising model 
performance.

The code cell below verifies conditions 1 and 2, confirming 15,906 students across 
the four valid response patterns.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('hsls_17_student_pets_sr_v1_0.csv')

In [2]:
# Define response indicators for each wave
by_resp  = df['X1SQSTAT'].isin([1, 2, 3]).astype(int)
f1_resp  = df['X2SQSTAT'].isin([1, 2, 3, 4]).astype(int)
u13_resp = df['X3SQSTAT'].isin([1, 2, 3, 4, 5]).astype(int)
f2_resp  = df['X4SQSTAT'].isin([1, 2, 3, 4]).astype(int)

# Valid outcome — X4EVERDROP must be 0 or 1, not -9
valid_outcome = df['X4EVERDROP'].isin([0, 1])

# Define the four valid patterns with valid outcome condition added
pattern_1111 = (by_resp==1) & (f1_resp==1) & (u13_resp==1) & (f2_resp==1) & valid_outcome
pattern_1101 = (by_resp==1) & (f1_resp==1) & (u13_resp==0) & (f2_resp==1) & valid_outcome
pattern_1011 = (by_resp==1) & (f1_resp==0) & (u13_resp==1) & (f2_resp==1) & valid_outcome
pattern_1001 = (by_resp==1) & (f1_resp==0) & (u13_resp==0) & (f2_resp==1) & valid_outcome

counts = {
    '1-1-1-1': pattern_1111.sum(),
    '1-1-0-1': pattern_1101.sum(),
    '1-0-1-1': pattern_1011.sum(),
    '1-0-0-1': pattern_1001.sum(),
}

for pattern, count in counts.items():
    print(f"Pattern {pattern}: {count}")

total = sum(counts.values())
print(f"\nTotal: {total}")
assert total == 15906, f"Expected 15906, got {total}"

Pattern 1-1-1-1: 13280
Pattern 1-1-0-1: 1430
Pattern 1-0-1-1: 797
Pattern 1-0-0-1: 399

Total: 15906


The assertion confirms exactly 15,906 students satisfy both conditions across the four 
valid response patterns. These 15,906 students form the basis for the dropout status 
mapping below. Note that this is not yet the final analysis sample — a further 
exclusion is introduced after the dropout mapping.

## Dropout Status Across Survey Waves

Before continuing the data cleaning, we show how the 15,906 students satisfying 
conditions 1 and 2 are categorised at each survey wave. We track how dropout status 
develops across waves and how X4EVERDROP is constructed from prior wave variables.

Dropout is tracked in two forms throughout the study. A **traditional dropout** is a 
student who left high school without obtaining any credential. An **alternative 
completer** is a student who left traditional high school but obtained a GED or 
equivalent credential. Both are coded X4EVERDROP = 1 — from an early intervention 
perspective, both represent a student who left the system, and preventing either 
outcome is the goal.

---

### First Follow-Up (F1, Spring 2012)

At F1, dropout status is first captured in `X2DROPSTAT`, which classifies each 
student into one of six categories:

| Code | Category | Count | % of 15,906 |
|------|----------|------:|------------:|
| 0 | Not dropout and not alternative completer | 14,406 | 90.6% |
| 1 | Current dropout as of spring 2012 | 321 | 2.0% |
| 2 | Alternative completer (GED) | 99 | 0.6% |
| 3 | Prior reported dropout episode | 828 | 5.2% |
| 4 | Status unknown | 202 | 1.3% |
| 9 | Out of scope or ineligible | 50 | 0.3% |
| **Total** | | **15,906** | **100%** |

`X2EVERDROP` then collapses `X2DROPSTAT` together with direct student reports, 
parent-reported prior school absence episodes for the student, and school enrollment 
records into a binary confirmed/not confirmed indicator. Students with unknown status 
(code 4) and out of scope (code 9) are resolved conservatively to X2EVERDROP = 0 — 
no dropout confirmed — rather than X2EVERDROP = 1. The implication of this is that 
all students labelled X2EVERDROP = 1 have confirmed dropout episodes. This is ground 
truth. The 1a/1b split is derived directly from X2DROPSTAT:

| Category | Count | % of 15,906 |
|----------|------:|------------:|
| 1b. Traditional dropout (X2DROPSTAT = 1 or 3) | 1,149 | 7.2% |
| 1a. Alternative completer GED (X2DROPSTAT = 2) | 99 | 0.6% |
| **Total confirmed dropout at F1 (X2EVERDROP = 1)** | **1,248** | **7.9%** |
| Not yet confirmed dropout (X2EVERDROP = 0) | 14,658 | 92.1% |
| **Total** | **15,906** | **100%** |

---

### 2013 Update (U13)

At U13, `X3DROPSTAT` classifies each student's status as of 2013:

| Code | Category | Count | % of 15,906 |
|------|----------|------:|------------:|
| -8 | Non-respondent at U13 | 1,829 | 11.5% |
| 0 | Not dropout and not alternative completer | 11,951 | 75.1% |
| 1 | Spring 2013 dropout | 460 | 2.9% |
| 2 | Alternative completer (GED) | 399 | 2.5% |
| 3 | Prior reported dropout episode | 681 | 4.3% |
| 4 | Status unknown | 586 | 3.7% |
| **Total** | | **15,906** | **100%** |

`X3EVERDROP` is then constructed using the following deterministic logic:
```
If X2EVERDROP = 1 then X3EVERDROP = 1
else if X3DROPSTAT = 1 then X3EVERDROP = 1
else if X3DROPSTAT = 2 then X3EVERDROP = 1
else X3EVERDROP = 0
```

The `else if` structure means the 1,248 students already confirmed at F1 are carried 
forward automatically — X3DROPSTAT is never evaluated for them. Only students with 
X2EVERDROP = 0 can be newly added at U13. Students with unknown status (code 4) and 
non-respondents (-8) are resolved conservatively to X3EVERDROP = 0. 

Note: of the 399 students classified as alternative completers at U13 (X3DROPSTAT = 2), 199 were already confirmed dropouts at F1 (X2EVERDROP = 1) and are carried forward under the first condition of the construction logic. Of these, 76 were GED completers at F1 and 123 were traditional dropouts at F1 who subsequently obtained a GED before the U13 interview. Only the remaining 200 students with no prior confirmed dropout episode are counted as genuinely new additions at U13.

The 2a/2b split reflects only the genuinely new additions beyond what was already confirmed at F1:

| Category | Count | % of 15,906 |
|----------|------:|------------:|
| Carried forward from F1 (X2EVERDROP = 1) | 1,248 | 7.9% |
| 2b. New traditional dropout at U13 | 201 | 1.3% |
| 2a. New alternative completer GED at U13 | 200 | 1.3% |
| **Cumulative confirmed dropout (X3EVERDROP = 1)** | **1,649** | **10.4%** |
| Not yet confirmed dropout (X3EVERDROP = 0) | 14,257 | 89.6% |
| **Total** | **15,906** | **100%** |

`X3DROPSTAT` does not capture students who dropped out between F1 and U13 but 
re-enrolled before the U13 interview — these students would not appear as current 
dropouts and their prior episode goes unrecorded at this wave. However, `S4DROPOUTHS` 
at F2 asks students retrospectively whether they ever stopped attending high school 
for four weeks or more — covering the entire high school period. Students with brief 
dropout episodes between F1 and U13 who re-enrolled are therefore captured at F2.

---

### Second Follow-Up (F2, 2016)

X4EVERDROP is constructed using the following logic:
```
if X3EVERDROP = 1 or S4DROPOUTHS = 1 then X4EVERDROP = 1
else if X3EVERDROP = -9 or S4DROPOUTHS = -9 then X4EVERDROP = -9
else X4EVERDROP = 0
```
X4EVERDROP carries forward all 1,649 cumulative dropouts from X3EVERDROP and adds 
students newly reporting a dropout episode in the F2 interview via S4DROPOUTHS. This 
question was only administered to students with X3EVERDROP = 0 — students already 
confirmed as dropouts were skipped past it. A portion of F2 dropout responses were 
logically inferred from credential status (no credential, GED, or other equivalency) 
rather than directly self-reported. These are treated as confirmed observations as 
they are grounded in verified credential status at the time of the F2 interview. This 
interview took place approximately three years after the students' expected graduation date. 

The 3a/3b split is derived from X4EVERGED:

| Category | Count | % of 15,906 |
|----------|------:|------------:|
| Carried forward from U13 (X3EVERDROP = 1) | 1,649 | 10.4% |
| 3b. New traditional dropout at F2 | 657 | 4.1% |
| 3a. New alternative completer GED at F2 | 138 | 0.9% |
| **Total new at F2** | **795** | **5.0%** |
| **Total confirmed dropout (X4EVERDROP = 1)** | **2,444** | **15.4%** |
| Never dropped out (X4EVERDROP = 0) | 13,462 | 84.6% |
| **Total** | **15,906** | **100%** |

---

### Final Dropout Composition

The 2,444 confirmed dropouts across the 15,906 students break down as follows:

| Wave | Traditional dropout | GED completer | Wave total | Cumulative total |
|------|--------------------:|--------------:|-----------:|-----------------:|
| F1 (spring 2012) | 1,149 | 99 | 1,248 | 1,248 |
| U13 (2013) | 201 | 200 | 401 | 1,649 |
| F2 (2016) | 657 | 138 | 795 | 2,444 |
| **Total** | **2,007** | **437** | | **2,444** |

Note: the cumulative total of 2,444 includes 61 students whose dropout classification derives from statistical imputation at U13. These are addressed in the next cleaning step.

The code cell below verifies the above mapping of dropout statuses at the different levels.

In [3]:
# Create the pre-imputation sample
pre_imp = df[
    (df['X1SQSTAT'].isin([1, 2, 3])) &
    (df['X4EVERDROP'].isin([0, 1]))
].copy()

print(f"Pre-imputation sample: {len(pre_imp)}")
print(f"\n--- F1 VERIFICATION ---")
print(f"X2DROPSTAT value counts:")
print(pre_imp['X2DROPSTAT'].value_counts().sort_index())
print(f"\nX2EVERDROP value counts:")
print(pre_imp['X2EVERDROP'].value_counts().sort_index())

# Verify 1a/1b split
dropout_1b = pre_imp['X2DROPSTAT'].isin([1, 3]).sum()
dropout_1a = (pre_imp['X2DROPSTAT'] == 2).sum()
total_f1 = (pre_imp['X2EVERDROP'] == 1).sum()
not_dropout_f1 = (pre_imp['X2EVERDROP'] == 0).sum()

print(f"\n1b. Traditional dropout: {dropout_1b}")
print(f"1a. Alternative completer GED: {dropout_1a}")
print(f"Total confirmed dropout at F1: {dropout_1b + dropout_1a}")
assert dropout_1b + dropout_1a == total_f1, \
    f"1a/1b split does not match X2EVERDROP=1: {dropout_1b + dropout_1a} vs {total_f1}"
assert total_f1 + not_dropout_f1 == len(pre_imp), \
    f"F1 total does not match sample size"
print(f"Not yet confirmed dropout: {not_dropout_f1}")
print(f"Total: {total_f1 + not_dropout_f1}")
print("F1 assertions passed ✓")

Pre-imputation sample: 15906

--- F1 VERIFICATION ---
X2DROPSTAT value counts:
X2DROPSTAT
0    14406
1      321
2       99
3      828
4      202
9       50
Name: count, dtype: int64

X2EVERDROP value counts:
X2EVERDROP
0    14658
1     1248
Name: count, dtype: int64

1b. Traditional dropout: 1149
1a. Alternative completer GED: 99
Total confirmed dropout at F1: 1248
Not yet confirmed dropout: 14658
Total: 15906
F1 assertions passed ✓


In [4]:
print(f"--- U13 VERIFICATION ---")
print(f"X3DROPSTAT value counts:")
print(pre_imp['X3DROPSTAT'].value_counts().sort_index())
print(f"\nX3EVERDROP value counts:")
print(pre_imp['X3EVERDROP'].value_counts().sort_index())

# Verify 2a/2b split — only new additions beyond F1
new_2b = ((pre_imp['X2EVERDROP'] == 0) & (pre_imp['X3DROPSTAT'] == 1)).sum()
new_2a = ((pre_imp['X2EVERDROP'] == 0) & (pre_imp['X3DROPSTAT'] == 2)).sum()
carried_f1 = (pre_imp['X2EVERDROP'] == 1).sum()
total_u13 = (pre_imp['X3EVERDROP'] == 1).sum()
not_dropout_u13 = (pre_imp['X3EVERDROP'] == 0).sum()

print(f"\nCarried forward from F1: {carried_f1}")
print(f"2b. New traditional dropout at U13: {new_2b}")
print(f"2a. New alternative completer GED at U13: {new_2a}")
print(f"Cumulative confirmed dropout (X3EVERDROP=1): {total_u13}")
assert carried_f1 + new_2b + new_2a == total_u13, \
    f"U13 cumulative does not add up: {carried_f1 + new_2b + new_2a} vs {total_u13}"
assert total_u13 + not_dropout_u13 == len(pre_imp), \
    f"U13 total does not match sample size"
print(f"Not yet confirmed dropout: {not_dropout_u13}")
print(f"Total: {total_u13 + not_dropout_u13}")
print("U13 assertions passed ✓")

--- U13 VERIFICATION ---
X3DROPSTAT value counts:
X3DROPSTAT
-8     1829
 0    11951
 1      460
 2      399
 3      681
 4      586
Name: count, dtype: int64

X3EVERDROP value counts:
X3EVERDROP
0    14257
1     1649
Name: count, dtype: int64

Carried forward from F1: 1248
2b. New traditional dropout at U13: 201
2a. New alternative completer GED at U13: 200
Cumulative confirmed dropout (X3EVERDROP=1): 1649
Not yet confirmed dropout: 14257
Total: 15906
U13 assertions passed ✓


In [5]:
print(f"--- F2 VERIFICATION ---")
total_dropout_f2 = (pre_imp['X4EVERDROP'] == 1).sum()
never_dropout_f2 = (pre_imp['X4EVERDROP'] == 0).sum()

# New additions at F2 — not already confirmed at U13
new_3b = ((pre_imp['X3EVERDROP'] == 0) & 
          (pre_imp['X4EVERDROP'] == 1) & 
          (pre_imp['X4EVERGED'] != 1)).sum()
new_3a = ((pre_imp['X3EVERDROP'] == 0) & 
          (pre_imp['X4EVERDROP'] == 1) & 
          (pre_imp['X4EVERGED'] == 1)).sum()

print(f"Carried forward from U13 (X3EVERDROP=1): {total_u13}")
print(f"3b. New traditional dropout at F2: {new_3b}")
print(f"3a. New alternative completer GED at F2: {new_3a}")
print(f"Total new at F2: {new_3b + new_3a}")
print(f"Total confirmed dropout (X4EVERDROP=1): {total_dropout_f2}")
assert total_u13 + new_3b + new_3a == total_dropout_f2, \
    f"F2 total does not add up: {total_u13 + new_3b + new_3a} vs {total_dropout_f2}"
assert total_dropout_f2 + never_dropout_f2 == len(pre_imp), \
    f"F2 total does not match sample size"
print(f"Never dropped out (X4EVERDROP=0): {never_dropout_f2}")
print(f"Total: {total_dropout_f2 + never_dropout_f2}")
print("F2 assertions passed ✓")

--- F2 VERIFICATION ---
Carried forward from U13 (X3EVERDROP=1): 1649
3b. New traditional dropout at F2: 657
3a. New alternative completer GED at F2: 138
Total new at F2: 795
Total confirmed dropout (X4EVERDROP=1): 2444
Never dropped out (X4EVERDROP=0): 13462
Total: 15906
F2 assertions passed ✓


In [6]:
print(f"--- FINAL COMPOSITION ---")
print(f"Traditional dropouts total: {dropout_1b + new_2b + new_3b}")
print(f"GED completers total:       {dropout_1a + new_2a + new_3a}")
print(f"Total dropouts:             {total_dropout_f2}")
assert dropout_1b + new_2b + new_3b + dropout_1a + new_2a + new_3a == total_dropout_f2, \
    f"Final composition does not add up"
print(f"Never dropped out:          {never_dropout_f2}")
print(f"Sample total:               {total_dropout_f2 + never_dropout_f2}")
assert total_dropout_f2 + never_dropout_f2 == len(pre_imp), \
    f"Final total does not match sample size"
print("Final composition assertions passed ✓")

--- FINAL COMPOSITION ---
Traditional dropouts total: 2007
GED completers total:       437
Total dropouts:             2444
Never dropped out:          13462
Sample total:               15906
Final composition assertions passed ✓


In [7]:
already_dropout = ((pre_imp['X3DROPSTAT'] == 2) & (pre_imp['X2EVERDROP'] == 1)).sum()
new_ged = ((pre_imp['X3DROPSTAT'] == 2) & (pre_imp['X2EVERDROP'] == 0)).sum()
already_ged_at_f1 = ((pre_imp['X3DROPSTAT'] == 2) & (pre_imp['X2DROPSTAT'] == 2)).sum()
trad_dropout_then_ged = ((pre_imp['X3DROPSTAT'] == 2) & (pre_imp['X2EVERDROP'] == 1) & (pre_imp['X2DROPSTAT'] != 2)).sum()

print(f"X3DROPSTAT=2 total: {(pre_imp['X3DROPSTAT']==2).sum()}")
print(f"Of those, already GED at F1 (X2DROPSTAT=2): {already_ged_at_f1}")
print(f"Of those, trad dropout at F1, now GED at U13: {trad_dropout_then_ged}")
print(f"Of those, already X2EVERDROP=1 total: {already_dropout}")
print(f"Of those, genuinely new at U13: {new_ged}")

X3DROPSTAT=2 total: 399
Of those, already GED at F1 (X2DROPSTAT=2): 76
Of those, trad dropout at F1, now GED at U13: 123
Of those, already X2EVERDROP=1 total: 199
Of those, genuinely new at U13: 200


All counts match the tables above. The cumulative dropout total reaches 2,444 across the three waves, consistent with the pre-imputation sample of 15,906 students.

## Condition 3 — Excluding Statistically Imputed Dropout Classifications

### Tracing Imputation Through the Target Variable Chain

To ensure the integrity of `X4EVERDROP`, we systematically traced all variables that 
contribute to its construction — both directly and indirectly. The full chain is:
```
X3HSCRED ──────────────────────────┐
X3HSCREDTYPE ──────────────────────┼──► X3DROPSTAT ──► X3EVERDROP ──► X4EVERDROP
X3LASTHSDATE ──────────────────────┘

S2DROPOUTHS ───────────────────────────────────────────────────────────────────────┐
P1DROPOUT ─────────────────────────────────────────────────────────────────────────┤
P2DROPOUTHS ───────────────────────────────────────────────────────────────────────┤──► 
X2ENROLSTAT ───────────────────────────────────────────────────────────────────────┘

──► X2EVERDROP ──► X3EVERDROP ──► X4EVERDROP

S4DROPOUTHS ──► X4EVERDROP
```

For each variable in this chain, we read the codebook entry in full and examined 
whether statistical imputation was applied. The findings are as follows:

- `X2ENROLSTAT` — deterministic, based on enrollment records. No imputation flag exists.
- `S2DROPOUTHS` — direct student self-report. No imputation flag exists.
- `P1DROPOUT` — direct parent report about the student. No imputation flag exists.
- `P2DROPOUTHS` — direct parent report about the student. No imputation flag exists.
- `X2DROPSTAT` — deterministic SAS logic from `X2ENROLSTAT`. No imputation flag exists.
- `X2EVERDROP` — deterministic combination of confirmed survey responses. No imputation 
  flag exists. The codebook explicitly states: "Yes value is known to be a true statement."
- `X3HSCRED` — imputed version of `S3HSCRED`. `X3HSCRED_IM` exists: 2 students imputed.
- `X3HSCREDTYPE` — imputed. `X3HSCREDTY_IM` exists: 26 students imputed in pre-imputation sample.
- `X3LASTHSDATE` — imputed version of `S3LASTHSDATE`. `X3LASTHSD_IM` exists: 147 students 
  imputed in pre-imputation sample.
- `X3DROPSTAT` — constructed from the above three inputs. Where any input was imputed and 
  materially affected the classification, `X3DROPSTAT_IM = 2`. This affects 88 students 
  in the pre-imputation sample.
- `X3EVERDROP` — constructed deterministically from `X2EVERDROP` and `X3DROPSTAT`. 
  Inherits imputation from `X3DROPSTAT`. `X3EVERDROP_IM = 2` for 96 students in the 
  full sample, of which 83 are in the pre-imputation sample.
- `S4DROPOUTHS` — direct student self-report at F2. Some values logically inferred from 
  confirmed credential status via `S4DROPOUTHS_I`. Logical inference from verified 
  credential status is not statistical imputation. No imputation flag exists.
- `X4EVERDROP` — the codebook explicitly states: "corresponding variable X3EVERDROP 
  included imputed values and those values were carried over to X4EVERDROP." No new 
  imputation is introduced at F2 — the only imputed values in `X4EVERDROP` are those 
  inherited from `X3EVERDROP` via `X3DROPSTAT`.

The conclusion is unambiguous: **`X3DROPSTAT` is the only point in the entire chain 
where statistical imputation materially affects dropout classification**, and 
`X3DROPSTAT_IM` is the precise flag identifying exactly which students are affected.

---

### Why These Students Are Excluded

167 students in the pre-imputation sample had at least one imputed input feeding into 
`X3DROPSTAT`. However, `X3DROPSTAT_IM` correctly identifies only the 88 students where 
the imputed input was the deciding factor in the classification — for the remaining 79, 
the imputed input was present but irrelevant because an earlier condition in the SAS 
logic could be satisfied using non-imputed data alone.

Of the 88 students with `X3DROPSTAT_IM = 2`, 83 were resolved to `X3EVERDROP = 1` and 
therefore `X4EVERDROP = 1`, and 5 were resolved to `X3EVERDROP = 0` and therefore 
`X4EVERDROP = 0`. Both groups are excluded.

The argument for excluding both groups rests on the same principle: a classification 
that rests on statistically estimated inputs introduces uncertainty that cannot be 
resolved from the public-use file. For the 83 students coded `X4EVERDROP = 1`, we 
cannot confirm they truly dropped out. For the 5 students coded `X4EVERDROP = 0`, we 
cannot confirm they truly did not drop out. In both cases, the imputed classification 
adds noise to the target variable — it creates observations where the label is less 
certain than for the remaining 15,818 students. Since the number of affected students 
is small (88 out of 15,906, or 0.55%), the cost of exclusion is negligible while the 
benefit — a target variable composed entirely of confirmed, non-estimated 
classifications — is substantial.

---

### Condition 3

**Condition 3 — Non-imputed dropout classification:** The student must have 
`X3DROPSTAT_IM = 0`, meaning no imputed input materially affected their dropout 
classification at U13. Students with `X3DROPSTAT_IM = 2` are excluded regardless of 
which direction their classification resolved.

Applying condition 3 reduces the sample from 15,906 to the **final analysis sample 
of 15,818 students**: 13,458 confirmed non-dropouts (85.1%) and 2,360 confirmed 
dropouts (14.9%).

The code cell below verifies condition 3 and confirms the final analysis sample.

In [8]:
# X3DROPSTAT_IM is the correct flag — it identifies only students where
# imputation materially affected the classification
print(f"\nX3DROPSTAT_IM value counts:")
print(pre_imp['X3DROPSTAT_IM'].value_counts().sort_index())

n_imputed = (pre_imp['X3DROPSTAT_IM'] == 2).sum()
n_not_imputed = (pre_imp['X3DROPSTAT_IM'] == 0).sum()
print(f"\nImputed (X3DROPSTAT_IM=2): {n_imputed}")
print(f"Not imputed (X3DROPSTAT_IM=0): {n_not_imputed}")
assert n_imputed + n_not_imputed == len(pre_imp), \
    f"Imputation flag counts do not sum to sample size"
assert n_imputed == 88, \
    f"Expected 88 imputed students, got {n_imputed}"
print("Imputation count assertion passed ✓")

# Show how the 88 imputed students resolved across X3EVERDROP
imputed = pre_imp[pre_imp['X3DROPSTAT_IM'] == 2]
print(f"\nOf the 88 imputed students, X3EVERDROP distribution:")
print(imputed['X3EVERDROP'].value_counts().sort_index())
print(f"\nOf the 88 imputed students, X4EVERDROP distribution:")
print(imputed['X4EVERDROP'].value_counts().sort_index())

# Apply condition 3 — remove all students where imputation materially
# affected X3DROPSTAT, regardless of which direction it resolved.
# This ensures both X4EVERDROP=1 and X4EVERDROP=0 are free from
# classifications that rest on statistically estimated inputs.
analysis_sample = pre_imp[pre_imp['X3DROPSTAT_IM'] == 0].copy()

print(f"\n=== ANALYSIS SAMPLE AFTER CONDITION 3 ===")
print(f"Pre-imputation sample:    {len(pre_imp)}")
print(f"Imputed students removed: {n_imputed}")
print(f"Analysis sample:          {len(analysis_sample)}")
assert len(analysis_sample) == 15818, \
    f"Expected 15818, got {len(analysis_sample)}"

print(f"\nX4EVERDROP distribution in analysis sample:")
print(analysis_sample['X4EVERDROP'].value_counts().sort_index())
dropout_rate = analysis_sample['X4EVERDROP'].mean() * 100
print(f"\nDropout rate: {round(dropout_rate, 1)}%")
print("Analysis sample assertion passed ✓")


X3DROPSTAT_IM value counts:
X3DROPSTAT_IM
0    15818
2       88
Name: count, dtype: int64

Imputed (X3DROPSTAT_IM=2): 88
Not imputed (X3DROPSTAT_IM=0): 15818
Imputation count assertion passed ✓

Of the 88 imputed students, X3EVERDROP distribution:
X3EVERDROP
0     5
1    83
Name: count, dtype: int64

Of the 88 imputed students, X4EVERDROP distribution:
X4EVERDROP
0     4
1    84
Name: count, dtype: int64

=== ANALYSIS SAMPLE AFTER CONDITION 3 ===
Pre-imputation sample:    15906
Imputed students removed: 88
Analysis sample:          15818

X4EVERDROP distribution in analysis sample:
X4EVERDROP
0    13458
1     2360
Name: count, dtype: int64

Dropout rate: 14.9%
Analysis sample assertion passed ✓


The code above confirms that 88 students in the pre-imputation sample of 15,906 have 
`X3DROPSTAT_IM = 2`, meaning statistical imputation materially affected their dropout 
classification at U13. Of these, 83 were resolved to `X3EVERDROP = 1` and 5 to 
`X3EVERDROP = 0`. At the F2 wave, 84 are coded `X4EVERDROP = 1` and 4 as 
`X4EVERDROP = 0` — the difference of one student reflects a case where an imputed 
non-dropout classification at U13 was later overridden by a confirmed dropout report 
at F2 via `S4DROPOUTHS`. All 88 students are excluded under condition 3.

The final analysis sample is **15,818 students**: 13,458 confirmed non-dropouts 
(85.1%) and 2,360 confirmed dropouts (14.9%). Every `X4EVERDROP = 1` in this sample 
derives from a confirmed survey response — no statistical estimation has contributed 
to any dropout classification in the analysis sample.

## Column Selection — Retaining Base Year Features Only

With the analysis sample of 15,818 students confirmed, we now reduce the dataset to 
only the columns relevant to the study. The raw file contains 9,614 columns spanning 
all survey waves, postsecondary records, weights, and administrative variables — the 
vast majority of which are irrelevant to a base year prediction model.

The following columns are retained:

- **STU_ID** — student identifier, retained for traceability
- **Base year feature columns** — all variables with prefixes X1, S1, P1, M1, A1, 
  and C1, including RUF-suppressed columns (coded -5) and imputation flag columns 
  (suffix _IM). RUF-suppressed columns are retained because the -5 value itself 
  carries information — it distinguishes students whose data was suppressed from 
  students who simply did not respond. Imputation flags are retained as metadata 
  for potential sensitivity analyses during modelling.
- **X4EVERDROP** — the target variable

All other columns are dropped — this includes all F1 (X2, S2, P2), U13 (X3, S3), 
and F2 variables except X4EVERDROP, all survey weights (W prefix), survey design 
variables (STRAT_ID, PSU), postsecondary and administrative variables, and the 
response pattern helper columns created during sample construction.

The code cell below applies this selection to the analysis sample.

In [9]:
# Define columns to keep
base_year_prefixes = ('X1', 'S1', 'P1', 'M1', 'A1', 'C1')

base_year_cols = [
    col for col in df.columns 
    if col.startswith(base_year_prefixes)
]

# Add STU_ID and target variable
cols_to_keep = ['STU_ID'] + base_year_cols + ['X4EVERDROP']

# Apply to analysis sample
clean_df = analysis_sample[cols_to_keep].copy()

print(f"Columns kept: {len(clean_df.columns)}")
print(f"Rows: {len(clean_df)}")
print(f"\nColumn breakdown:")
print(f"  STU_ID: 1")
print(f"  Base year (X1): {sum(1 for c in clean_df.columns if c.startswith('X1'))}")
print(f"  Base year (S1): {sum(1 for c in clean_df.columns if c.startswith('S1'))}")
print(f"  Base year (P1): {sum(1 for c in clean_df.columns if c.startswith('P1'))}")
print(f"  Base year (M1): {sum(1 for c in clean_df.columns if c.startswith('M1'))}")
print(f"  Base year (A1): {sum(1 for c in clean_df.columns if c.startswith('A1'))}")
print(f"  Base year (C1): {sum(1 for c in clean_df.columns if c.startswith('C1'))}")
print(f"  X4EVERDROP: 1")
print(f"\nX4EVERDROP distribution:")
print(clean_df['X4EVERDROP'].value_counts().sort_index())

Columns kept: 1291
Rows: 15818

Column breakdown:
  STU_ID: 1
  Base year (X1): 191
  Base year (S1): 286
  Base year (P1): 199
  Base year (M1): 150
  Base year (A1): 266
  Base year (C1): 197
  X4EVERDROP: 1

X4EVERDROP distribution:
X4EVERDROP
0    13458
1     2360
Name: count, dtype: int64


The clean dataset contains 15,818 students and 1,291 columns — 1 identifier, 1,289 
base year feature columns, and the target variable X4EVERDROP. The column breakdown 
across respondent types is: 191 X1 composites, 286 S1 student instrument, 199 P1 
parent instrument, 150 M1 math teacher, 266 A1 administrator, and 197 C1 counselor 
variables. This dataset is saved as the input for all subsequent modelling notebooks.

## Recoding Missing Values

All negative codes in the HSLS:09 dataset represent forms of missingness rather than 
meaningful numeric values. The full set of missing codes is:

| Code | Meaning |
|------|---------|
| -1 | Item skipped |
| -2 | Refused (rare) |
| -3 | Not ascertained (rare) |
| -4 | Nonrespondent |
| -5 | Suppressed (RUF only) |
| -6 | Component not applicable |
| -7 | Item legitimate skip |
| -8 | Unit non-response |
| -9 | Missing |

If left as integers, any model would treat these as meaningful numeric values — for 
example interpreting -5 as lower than -4 and higher than -6. This is incorrect and 
would introduce substantial noise into every model. All negative codes must be 
recoded to NaN (Not a Number) — pandas' standard representation of a missing value 
— before any modelling takes place.

The imputation flag columns (_IM suffix) are exempt from this recoding as they only 
contain 0 and 1 and do not use negative codes. STU_ID and X4EVERDROP are also exempt 
— STU_ID contains no missing codes, and X4EVERDROP has already been filtered to 
contain only 0 and 1 in the analysis sample.

The code cell below applies the recode to all feature columns, then verifies that no 
negative values remain in the dataset.

In [13]:
# Columns to recode — all except STU_ID, X4EVERDROP, and _IM flag columns
exempt_cols = ['STU_ID', 'X4EVERDROP']
im_cols = [col for col in clean_df.columns if col.endswith('_IM')]
cols_to_recode = [
    col for col in clean_df.columns 
    if col not in exempt_cols and col not in im_cols
]

missing_codes = [-1, -2, -3, -4, -5, -6, -7, -8, -9]
missing_codes_float = {float(x) for x in missing_codes}

# Split into integer and float columns
int_cols = [col for col in cols_to_recode 
            if clean_df[col].dtype in ['int64', 'int32']]
float_cols_all = [col for col in cols_to_recode 
                  if clean_df[col].dtype in ['float64', 'float32']]

# For float columns, identify which ones have legitimate negatives below -9
# These need special handling — only recode exact integer codes
legit_negative_float_cols = [col for col in float_cols_all 
                              if clean_df[col].min() < -9]
regular_float_cols = [col for col in float_cols_all 
                      if col not in legit_negative_float_cols]

print(f"Integer columns to recode: {len(int_cols)}")
print(f"Float columns with only missing-code negatives: {len(regular_float_cols)}")
print(f"Float columns with legitimate negatives below -9: {len(legit_negative_float_cols)}")
print(f"Legitimate negative float columns: {legit_negative_float_cols}")

# Count negatives before
neg_before = (clean_df[cols_to_recode] < 0).sum().sum()
print(f"\nNegative values before recoding: {neg_before}")

# Recode integer columns — replace all missing codes
clean_df = clean_df.copy()
for col in int_cols:
    clean_df[col] = clean_df[col].replace(missing_codes, np.nan)

# Recode regular float columns — replace exact float versions of missing codes
for col in regular_float_cols:
    clean_df[col] = clean_df[col].replace(
        {float(c): np.nan for c in missing_codes}
    )

# Recode legitimate negative float columns — ONLY replace exact integer codes
# Leave all other negatives (e.g. -0.53, -1.74) untouched
for col in legit_negative_float_cols:
    clean_df[col] = clean_df[col].replace(
        {float(c): np.nan for c in missing_codes}
    )

# Verify — check no exact missing codes remain in integer columns
print(f"\nVerifying no missing codes remain...")
for code in missing_codes:
    count_int = (clean_df[int_cols] == code).sum().sum()
    count_float = (clean_df[regular_float_cols] == float(code)).sum().sum()
    if count_int > 0 or count_float > 0:
        print(f"WARNING: code {code} still present — int: {count_int}, float: {count_float}")

# Verify legitimate negatives are preserved
for col in legit_negative_float_cols:
    remaining_neg = (clean_df[col] < -9).sum()
    print(f"{col}: {remaining_neg} legitimate negatives preserved")

# Show missingness summary
total_cells = clean_df[cols_to_recode].size
missing_cells = clean_df[cols_to_recode].isna().sum().sum()
print(f"\nTotal feature cells: {total_cells:,}")
print(f"Missing (NaN) cells: {missing_cells:,}")
print(f"Missing rate: {round(missing_cells / total_cells * 100, 1)}%")

assert clean_df['X4EVERDROP'].isin([0, 1]).all()
assert clean_df['STU_ID'].isna().sum() == 0
print("\nX4EVERDROP and STU_ID integrity confirmed ✓")

Integer columns to recode: 1205
Float columns with only missing-code negatives: 55
Float columns with legitimate negatives below -9: 0
Legitimate negative float columns: []

Negative values before recoding: 9295590

Verifying no missing codes remain...

Total feature cells: 19,930,680
Missing (NaN) cells: 9,023,014
Missing rate: 45.3%

X4EVERDROP and STU_ID integrity confirmed ✓


In [14]:
continuous_cols = ['X1SES', 'X1TXMTH', 'X1SES_U']
for col in continuous_cols:
    if col in clean_df.columns:
        neg_count = (clean_df[col] < 0).sum()
        col_min = clean_df[col].min()
        print(f"{col}: {neg_count} negative values, min = {col_min:.4f}")

X1SES: 7598 negative values, min = -1.9302
X1TXMTH: 7173 negative values, min = -2.5672
X1SES_U: 7636 negative values, min = -1.9183


The missing value recode completed successfully. All negative NCES missing codes (-1 
through -9) have been replaced with NaN across all 1,205 integer columns and 55 float 
columns. Continuous variables with legitimate negative values — including `X1SES` 
(min -1.93), `X1TXMTH` (min -2.57), and `X1SES_U` (min -1.92) — are confirmed 
intact with their negative values preserved. The overall missing rate of 45.3% across 
the feature space is expected given the survey design: RUF-suppressed columns are 
entirely missing on the public-use file, and many questions were only administered to 
subgroups of students. This missingness will be addressed during feature selection in 
the modelling notebooks. `X4EVERDROP` and `STU_ID` are confirmed unaffected.

## Checking for Duplicate Rows and Constant Columns

Two final checks before saving the clean dataset.

**Duplicate rows** — each row should represent a unique student identified by 
`STU_ID`. A duplicate would mean the same student appears twice, which would inflate 
their influence during model training.

**Constant columns** — a column where every student has the same value (or NaN) 
carries no predictive information and can be safely dropped before modelling.

The code cell below verifies both.

In [15]:
# Check for duplicate STU_IDs
n_duplicates = clean_df['STU_ID'].duplicated().sum()
print(f"Duplicate STU_IDs: {n_duplicates}")
assert n_duplicates == 0, f"Expected 0 duplicates, got {n_duplicates}"
print("No duplicate rows ✓")

# Check for constant columns — columns with only one unique non-NaN value
constant_cols = [
    col for col in cols_to_recode
    if clean_df[col].nunique(dropna=True) <= 1
]
print(f"\nConstant columns (zero variance): {len(constant_cols)}")
if len(constant_cols) > 0:
    print("Columns (showing first 20):")
    for col in constant_cols[:20]:
        print(f"  {col}: unique values = {clean_df[col].unique()}")
    if len(constant_cols) > 20:
        print(f"  ... and {len(constant_cols) - 20} more (all NaN)")
    clean_df = clean_df.drop(columns=constant_cols)
    print(f"\nDropped {len(constant_cols)} constant columns")
    print(f"Remaining columns: {len(clean_df.columns)}")
else:
    print("No constant columns found — no columns dropped ✓")

print(f"\nFinal dataset shape: {clean_df.shape}")

Duplicate STU_IDs: 0
No duplicate rows ✓

Constant columns (zero variance): 378
Columns (showing first 20):
  X1NCESID: unique values = [nan]
  X1ASIAN: unique values = [nan]
  X1PACISLE: unique values = [nan]
  X1AMINDIAN: unique values = [nan]
  X1HISPTYPE: unique values = [nan]
  X1ASIANTYPE: unique values = [nan]
  X1NATIVELANG: unique values = [nan]
  X1PAR1OCC6: unique values = [nan]
  X1PAR1OCC_STEM2: unique values = [nan]
  X1PAR2OCC6: unique values = [nan]
  X1PAR2OCC_STEM2: unique values = [nan]
  X1MOMOCC6: unique values = [nan]
  X1MOMOCC_STEM2: unique values = [nan]
  X1DADOCC6: unique values = [nan]
  X1DADOCC_STEM2: unique values = [nan]
  X1STU30OCC6: unique values = [nan]
  X1STU30OCC_STEM2: unique values = [nan]
  X1STUPRVSCHL_R: unique values = [nan]
  X1SQINCAPABL: unique values = [nan]
  X1CENDIV: unique values = [nan]
  ... and 358 more (all NaN)

Dropped 378 constant columns
Remaining columns: 913

Final dataset shape: (15818, 913)


No duplicate STU_IDs were found — each row represents a unique student. 378 constant 
columns were identified and dropped — all containing only NaN values. These are the 
RUF-suppressed variables that were entirely suppressed on the public-use file, 
leaving only -5 values which were recoded to NaN in the previous step. Retaining 
them through column selection was correct — their suppression status was only 
confirmed after the missing value recode. The final dataset contains 15,818 students 
and 913 columns: 1 student identifier, 911 base year feature columns with real data, 
and the target variable X4EVERDROP.

## Saving the Clean Dataset

The clean dataset is saved as a single CSV file. All subsequent modelling notebooks 
load from this file — no further cleaning or row filtering is required before 
modelling begins.

Throughout this cleaning process, particular care has been taken to ensure the 
integrity of the target variable. Every student coded X4EVERDROP = 1 in the analysis 
sample represents a confirmed instance where the student stopped attending school for 
a period of four weeks or more — verified through direct survey responses, parental 
reports, or confirmed credential status. No statistical estimates contribute to any 
dropout classification in this sample. This is the ground truth. There may be undiscovered 
dropout episodes among students coded X4EVERDROP = 0 — students who dropped out briefly between
waves, re-enrolled, and did not report the episode in the F2 interview — but every decision in 
this cleaning process has been designed to minimise that uncertainty as far as the public-use file allows.



In [16]:
clean_df.to_csv('hsls_clean.csv', index=False)
print(f"Saved: hsls_clean.csv")
print(f"Shape: {clean_df.shape}")
print(f"\nX4EVERDROP distribution:")
print(clean_df['X4EVERDROP'].value_counts().sort_index())
print(f"\nDropout rate: {round(clean_df['X4EVERDROP'].mean()*100, 1)}%")

Saved: hsls_clean.csv
Shape: (15818, 913)

X4EVERDROP distribution:
X4EVERDROP
0    13458
1     2360
Name: count, dtype: int64

Dropout rate: 14.9%


In [17]:
total_cells = clean_df.drop(columns=['STU_ID', 'X4EVERDROP']).size
missing_cells = clean_df.drop(columns=['STU_ID', 'X4EVERDROP']).isna().sum().sum()
print(f"Total feature cells: {total_cells:,}")
print(f"Missing cells: {missing_cells:,}")
print(f"Missing rate: {round(missing_cells / total_cells * 100, 1)}%")

# Breakdown by prefix
for prefix in ['X1', 'S1', 'P1', 'M1', 'A1', 'C1']:
    cols = [c for c in clean_df.columns if c.startswith(prefix)]
    if cols:
        missing = clean_df[cols].isna().sum().sum()
        total = clean_df[cols].size
        print(f"{prefix}: {len(cols)} columns, {round(missing/total*100, 1)}% missing")

Total feature cells: 14,410,198
Missing cells: 3,043,810
Missing rate: 21.1%
X1: 161 columns, 10.2% missing
S1: 271 columns, 10.7% missing
P1: 141 columns, 45.2% missing
M1: 134 columns, 32.4% missing
A1: 54 columns, 18.3% missing
C1: 150 columns, 20.2% missing


## Clean Dataset Summary

The clean dataset `hsls_clean.csv` is the input for all modelling notebooks. 
No further row filtering or structural cleaning is required.

**Sample:** 15,818 students — 13,458 confirmed non-dropouts (85.1%) and 2,360 
confirmed dropouts (14.9%). Every X4EVERDROP = 1 represents a verified dropout 
episode derived from confirmed survey responses with no statistical estimates.

**Features:** 911 base year columns across six respondent types:

| Prefix | Respondent | Columns | Missing rate |
|--------|------------|--------:|-------------:|
| X1 | Base year composites | 161 | 10.2% |
| S1 | Student instrument | 271 | 10.7% |
| P1 | Parent instrument | 141 | 45.2% |
| M1 | Math teacher instrument | 134 | 32.4% |
| A1 | Administrator instrument | 54 | 18.3% |
| C1 | Counselor instrument | 150 | 20.2% |
| **Total** | | **911** | **21.1%** |

The overall missing rate of 21.1% reflects the survey design — many questions were 
administered to subgroups only, and some variables had lower questionnaire response 
rates (particularly parent and teacher instruments). Missing values are represented 
as NaN throughout. 

**Steps taken in this notebook:**
1. Loaded raw file (23,503 students × 9,614 columns)
2. Applied Condition 1 — base year respondents only (21,444 students)
3. Applied Condition 2 — valid X4EVERDROP outcome, four response patterns (15,906 students)
4. Mapped dropout status across F1, U13 and F2 waves and verified all intermediate counts
5. Applied Condition 3 — excluded 88 students with statistically imputed dropout classifications (15,818 students)
6. Selected base year feature columns plus STU_ID and X4EVERDROP (1,291 columns)
7. Recoded all NCES missing codes (-1 through -9) to NaN, preserving legitimate continuous negatives
8. Verified no duplicate rows
9. Dropped 378 constant columns (all-NaN RUF-suppressed variables)
10. Saved clean dataset: 15,818 rows × 913 columns